<table>
  <tr>
    <td><div align="left"><font size="30">Discrete-time and hybrid systems</font></div></td>
    <td><img src="https://raw.githubusercontent.com/petercorke/bdsim/master/figs/BDSimLogo_NoBackgnd%402x.png" width="300"></td>
  </tr>
</table>

(c) Peter Corke 2026

In [ ]:
# Install bdsim.  In JupyterLite, piplite checks the local wheel
# (built with 'make wheel') before falling back to PyPI.
# In a regular Jupyter server this is a no-op (piplite is not available).
try:
    import piplite
    await piplite.install('bdsim')
except ImportError:
    pass  # already installed (classic Jupyter)


In this notebook we will build a model of a simple discrete-time system, a first-order system with a feedback loop, and simulate it.

Basic knowledge of `bdsim` from the "Getting started" notebook is assumed.

We start with the standard boilerplate: import the package, establish the simulation environment, create an empty block diagram to work with.

In [ ]:
import bdsim
sim = bdsim.BDSim()
bd = sim.blockdiagram()

We will create a discrete-time equivalent model of the system used in the "Getting started" notebook which was

![Block diagram sketch](../figs/bd1-sketch.png)

The first thing we need to for a discrete-time system is to create a clock source

In [ ]:
clock = bd.clock(10, "Hz")

`bdsim` does not have functionality to convert a continuous-time transfer function to discrete-time, but there are lots of packages that will do it for you (the first three are open source)


* Python Control [`control.sample_system`](https://python-control.readthedocs.io/en/latest/generated/control.sample_system.html)
* SciPy [`scipy.signal.cont2discrete`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.cont2discrete.html)
* Octave [`tf`](https://gnu-octave.github.io/pkg-control/@tf_tf.html) and [`c2d`](https://gnu-octave.github.io/pkg-control/@lti_c2d.html)
* MATLAB `tf` and `c2d`

For a sample rate of 10Hz and using the ZOH method for discretisation, we can write a discrete-time version of the plant as follows:

In [ ]:
plant = bd.LTI_SISO_S(clock, 0.02439, [1, - 0.9512], name="plant")

Note that in the continuous time case the block name was `LTI_SISO`.  

All `bdsim` discrete-time blocks have a `_S` suffix which denotes "**S**ampled data".  All blocks that have a `_S` suffix must have a clock as their first argument.


Now let's define the remaining blocks

In [ ]:
sum = bd.SUM("+-")
gain = bd.GAIN(10)
demand = bd.STEP(T=1, name="demand")
scope = bd.SCOPE(styles=["k", "r--"], stairs=True, loc="lower right")

One very subtle difference is the argument `stairs=True` to the `SCOPE` block.  Since this is a discrete-time system we should really plot the signals as piecewise constant.

In [ ]:
bd.connect(sum, gain)
bd.connect(plant, sum[1])
bd.connect(demand, sum[0], scope[1])
bd.connect(gain, plant)
bd.connect(plant, scope[0])

and we could also use the Pythonic operator overload approach introduced in the "Getting started" notebook.


Next we compile and check our model

In [ ]:
bd.compile(verbose=True)
bd.report_summary()


Now it's time to run our model.  We'll see what happens over the first 5 seconds.

In [ ]:
out = sim.run(bd, T=5)

We see a classic first-order response to the unit step at time $t=1$ and considerable steady state error.

The time series data from the simulation is available in the return value `out` but the format is a little different this time

In [ ]:
print(out)

`t` and `x` represent the continuous-time state but that is empty in this case, since there are no continuous time states.

Instead we look at the substructure for the clock object, `clock0` which contains a time vector `t` with 50 elements (5s at 10Hz) and a state vector with single column. According to `clock0.xnames` corresponds to "plant:X_0", the only state variable of our first-order plant.


<div class="alert alert-block alert-info">
<b>Note:</b> Note that our clock was assigned a default name of <code>clock.0</code> and the struct object holding our results uses a dot to separate fields.  Therefore the name of the clock field in this case is <code>clock0</code> not <code>clock.0</code>.
</div>

# A hybrid continuous-discrete example

We will use the same plant and controller as before, but add a zero-order hold (ZOH) to discretise the controller output (between the `GAIN` and the `LTI_SS` blocks).
The ZOH will be clocked at 10Hz.

In [ ]:
bd = sim.blockdiagram()

clock = bd.clock(5, "Hz")

zoh = bd.ZOH(clock)

plant = bd.LTI_SISO(0.5, [2, 1], name="plant")
sum = bd.SUM("+-")
gain = bd.GAIN(10)
demand = bd.STEP(T=1, name="demand")
scope = bd.SCOPE(styles=["k", "r--"], loc="lower right")

bd.connect(sum, gain)
bd.connect(plant, sum[1])
bd.connect(demand, sum[0], scope[1])
bd.connect(gain, zoh)
bd.connect(zoh, plant)
bd.connect(plant, scope[0])

bd.compile()
bd.report_summary()

Now we run the model

In [ ]:
out = sim.run(bd, T=5)

and the result looks fairly similar to before.  If we look at the simulation results we can see both the discrete-time and continuous time states

In [ ]:
print(out)

There is one continuous-time state vector `x[:,0]` corresponding to the plant, and one discrete-time state vector `clock1.X[:,0]` corresponding to the ZOH.  Each state vector has a corresponding time vector `t` and `clock1.t`.  

<div class="alert alert-block alert-info">
<b>Note:</b>The elements of <code>clock1.t</code> are regularly spaced because that subsystem is driven by a 5Hz clock.  The elements of <code>t</code> will be non-uniformly spaced and determined by the numerical integrator.  Regular sampling can be achieved by passing the <code>dt</code> option to <code>sim.run(...)</code>.
</div>


